In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__}")
print(f"device: {device}")
if device.type == "cuda":
    print(f"gpu: {torch.cuda.get_device_name(0)}")
    print(f"vram: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch 2.14.0+cu130
device: cuda
gpu: NVIDIA GeForce RTX 4060 Laptop GPU
vram: 8.6 GB


In [4]:
# Tensor ops on GPU: allocate, move, and time a matmul
import time

a = torch.randn(4096, 4096, device=device)
b = torch.randn(4096, 4096, device=device)

if device.type == "cuda":
    torch.cuda.synchronize()
start = time.time()
c = a @ b
if device.type == "cuda":
    torch.cuda.synchronize()
elapsed = time.time() - start

print(f"matmul 4096x4096 on {device}: {elapsed*1000:.2f} ms")
print(f"result shape: {c.shape}, dtype: {c.dtype}")

matmul 4096x4096 on cuda: 82.47 ms
result shape: torch.Size([4096, 4096]), dtype: torch.float32


In [5]:
# Minimal training loop: tiny MLP fit to a toy regression, on GPU
import torch.nn as nn

torch.manual_seed(0)

model = nn.Sequential(
    nn.Linear(10, 64),
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 1),
).to(device)

X = torch.randn(1024, 10, device=device)
y = X.sum(dim=1, keepdim=True)  # target: sum of inputs

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(200):
    optimizer.zero_grad()
    pred = model(X)
    loss = loss_fn(pred, y)
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0:
        print(f"epoch {epoch:4d}  loss {loss.item():.4f}")

print(f"final loss: {loss.item():.4f}")

epoch    0  loss 9.5799
epoch   50  loss 0.6638
epoch  100  loss 0.0983
epoch  150  loss 0.0435
final loss: 0.0283
